<a href="https://colab.research.google.com/github/parika8ec-hub/DataScience_Project_BIA/blob/Labs/Lesson_11_Lab_Image_Classification_using_scikit_learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Image Classification using `sklearn.svm`**
Image files used are from https://github.com/Abhishek-Arora/Image-Classification-Using-SVM



**Try other ML algorithms for classsification and compare the performance**


### Image Classification using SVM

This notebook demonstrates how to perform image classification using Support Vector Machines (SVM) in scikit-learn.

**The main goal is to build a model that can accurately classify images into different categories.**

The notebook covers the following tasks:

1. **Loading and Preprocessing Image Data:** Images are loaded from a structured directory and preprocessed to a consistent size.
2. **Splitting Data into Training and Testing Sets:** The dataset is split to evaluate the model's performance on unseen data.
3. **Training an SVM Classifier with Parameter Optimization:** An SVM classifier is trained, and its hyperparameters are tuned using GridSearchCV to find the best configuration.
4. **Making Predictions:** The trained model is used to predict the categories of images in the testing set.
5. **Evaluating Model Performance:** The performance of the classifier is assessed using a classification report, which provides metrics such as precision, recall, and F1-score.

### By the end of this exercise, you will have a working image classification model and understand how to evaluate its performance.

### **Try other ML algorithms for classsification and compare the performance**

In [1]:
!pip install gitpython --quiet # Install the library to clone git repos

import git

# Clone the repository
repo_url = 'https://github.com/Abhishek-Arora/Image-Classification-Using-SVM.git'
repo_dir = 'Image-Classification-Using-SVM'
git.Repo.clone_from(repo_url, repo_dir, branch='master')

# Update the image_folder path
image_folder = "Image-Classification-Using-SVM/images"  # New path to the image folder

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
%matplotlib notebook
from sklearn import svm, metrics, datasets
from sklearn.utils import Bunch
from sklearn.model_selection import GridSearchCV, train_test_split

from skimage.io import imread
from skimage.transform import resize
import skimage.io # Import the skimage module

In [3]:
import os

image_folder = "/content/Image-Classification-Using-SVM/images"  # Path to the image folder
categories = os.listdir(image_folder)  # Get the list of categories

for category in categories:
  category_path = os.path.join(image_folder, category)
  for image_file in os.listdir(category_path):
    image_path = os.path.join(category_path, image_file)
    # Process the image file (e.g., read using OpenCV or Pillow)

### Load images in structured directory like it's sklearn sample dataset

In [4]:
def load_image_files(container_path, dimension=(64, 64)):
    """
    Load image files with categories as subfolder names
    which performs like scikit-learn sample dataset

    Parameters
    ----------
    container_path : string or unicode
        Path to the main folder holding one subfolder per category
    dimension : tuple
        size to which image are adjusted to

    Returns
    -------
    Bunch
    """
    image_dir = Path(container_path)
    folders = [directory for directory in image_dir.iterdir() if directory.is_dir()]
    categories = [fo.name for fo in folders]

    descr = "A image classification dataset"
    images = []
    flat_data = []
    target = []
    for i, direc in enumerate(folders):
        for file in direc.iterdir():
            img = skimage.io.imread(file)
            img_resized = resize(img, dimension, anti_aliasing=True, mode='reflect')
            flat_data.append(img_resized.flatten())
            images.append(img_resized)
            target.append(i)
    flat_data = np.array(flat_data)
    target = np.array(target)
    images = np.array(images)

    return Bunch(data=flat_data,
                 target=target,
                 target_names=categories,
                 images=images,
                 DESCR=descr)

In [5]:
# Update the path to where your images are located
image_dataset = load_image_files("/content/Image-Classification-Using-SVM/images/")

### Split data

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    image_dataset.data, image_dataset.target, test_size=0.3,random_state=109)

### Train data with parameter optimization

In [7]:
param_grid = [
  {'C': [1, 10, 100, 1000], 'kernel': ['linear']},
  {'C': [1, 10, 100, 1000], 'gamma': [0.001, 0.0001], 'kernel': ['rbf']},
 ]
svc = svm.SVC()
clf = GridSearchCV(svc, param_grid)
clf.fit(X_train, y_train)

GridSearchCV(estimator=SVC(),
             param_grid=[{'C': [1, 10, 100, 1000], 'kernel': ['linear']},
                         {'C': [1, 10, 100, 1000], 'gamma': [0.001, 0.0001],
                          'kernel': ['rbf']}])

### Predict

In [8]:
y_pred = clf.predict(X_test)

### Report

In [9]:
print("Classification report for - \n{}:\n{}\n".format(
    clf, metrics.classification_report(y_test, y_pred)))

Classification report for - 
GridSearchCV(estimator=SVC(),
             param_grid=[{'C': [1, 10, 100, 1000], 'kernel': ['linear']},
                         {'C': [1, 10, 100, 1000], 'gamma': [0.001, 0.0001],
                          'kernel': ['rbf']}]):
              precision    recall  f1-score   support

           0       1.00      0.75      0.86        16
           1       0.53      0.48      0.50        21
           2       0.36      0.31      0.33        16
           3       0.72      0.76      0.74        17
           4       0.77      1.00      0.87        23

    accuracy                           0.68        93
   macro avg       0.67      0.66      0.66        93
weighted avg       0.67      0.68      0.67        93


